In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE      = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
BASE_IN   = BASE / "data/processed/base_analitica_setorial_2023.csv"
HORAS_IN  = BASE / "data/processed/horas_por_setor_2023_corrigido.csv"
PIB_IN    = BASE / "data/processed/pib_setorial_2012_2023.csv"

print("Imports OK")

In [ ]:
df_base  = pd.read_csv(BASE_IN)
df_horas = pd.read_csv(HORAS_IN, index_col=0)
df_pib   = pd.read_csv(PIB_IN)

pib_2023     = df_pib[df_pib["ano"] == 2023].squeeze()
pib_total_rs = pib_2023["pib_total"] * 1e6
df_sim       = df_base[df_base["incluir"] == True].copy()

In [ ]:
# Mesma faixa da Formulação A, mas o ΔH médio é calculado trabalhador a trabalhador.
# A ideia: reduzir de 42h para 40h não é a mesma coisa que reduzir de 44h para 40h,
# então tirar a média das reduções individuais é mais honesto do que reduzir a média da faixa.
# Assumi distribuição uniforme em 41, 42, 43, 44 — provavelmente errado, mas auditável.

horas_faixa = np.array([41, 42, 43, 44])
delta_h_por_hora = (40 - horas_faixa) / horas_faixa

delta_h_C = delta_h_por_hora.mean()
delta_h_A = (40 - 42) / 42  

print("── Comparação ΔH: Formulação A vs C (escopo idêntico) ──")
print(f"\nFormulação A: ΔH = {delta_h_A*100:.4f}%")
print(f"  (redução da média da faixa: 42h → 40h)")
print(f"\nFormulação C: ΔH = {delta_h_C*100:.4f}%")
print(f"  (média ponderada uniforme: ΔH(41)+ΔH(42)+ΔH(43)+ΔH(44) / 4)")
print(f"\nDecomposição ΔH por hora:")
for h, dh in zip(horas_faixa, delta_h_por_hora):
    print(f"  h={h}h: ΔH = {dh*100:.4f}%")
print(f"\nDiferença A vs C: {abs(delta_h_C - delta_h_A)*100:.4f} pp")
print(f"Viés relativo da Formulação A: {abs(delta_h_A/delta_h_C - 1)*100:.2f}%")

In [ ]:
SEMANAS_ANO = 52

resultados_comp = []

for _, row in df_sim.iterrows():
    setor   = row["setor"]
    vab_rs  = row["vab_r$_bi"] * 1e9
    parcela = row["pct_41_44h"] / 100.0  

    # Impacto Formulação A
    delta_vab_A = vab_rs * parcela * delta_h_A

    # Impacto Formulação C
    delta_vab_C = vab_rs * parcela * delta_h_C

    resultados_comp.append({
        "setor":              setor,
        "pct_41_44h":         round(row["pct_41_44h"], 2),
        "vab_r$_bi":          round(vab_rs / 1e9, 2),
        "delta_vab_A_r$_bi":  round(delta_vab_A / 1e9, 3),
        "delta_vab_C_r$_bi":  round(delta_vab_C / 1e9, 3),
        "diferenca_r$_bi":    round((delta_vab_C - delta_vab_A) / 1e9, 3),
        "diferenca_pct":      round((delta_vab_C / delta_vab_A - 1) * 100, 2)
                              if delta_vab_A != 0 else None,
    })

df_comp = pd.DataFrame(resultados_comp)

print("\n── Comparação setorial: Formulação A vs C ───────────────")
print(df_comp.to_string(index=False))


total_A = df_comp["delta_vab_A_r$_bi"].sum()
total_C = df_comp["delta_vab_C_r$_bi"].sum()
pib_A   = total_A * 1e9 / pib_total_rs * 100
pib_C   = total_C * 1e9 / pib_total_rs * 100

print(f"\n── Resultado agregado ───────────────────────────────────")
print(f"Formulação A: ΔVAB = R$ {total_A:.3f} bi  →  ΔPIB = {pib_A:.4f}%")
print(f"Formulação C: ΔVAB = R$ {total_C:.3f} bi  →  ΔPIB = {pib_C:.4f}%")
print(f"Diferença:    ΔVAB = R$ {total_C - total_A:.3f} bi  →  ΔPIB = {pib_C - pib_A:.4f} pp")
print(f"\nViés relativo da Formulação A vs C: {abs(pib_A/pib_C - 1)*100:.2f}%")

In [ ]:
print("""
── INTERPRETAÇÃO ────────────────────────────────────────

Formulação A usa a média da faixa (42h) como ponto de referência.
Formulação C usa a média ponderada das reduções individuais dentro
da faixa 41–44h, assumindo distribuição uniforme.

Diferença entre A e C:
  - Se pequena (< 5%): A é aproximação adequada.
  - Se média (5–15%): A introduz viés sistemático moderado.
  - Se grande (> 15%): A é inadequada; C deve ser adotada.

Limitação remanescente da Formulação C:
  A distribuição uniforme dentro de 41–44h é uma aproximação.
  A distribuição real provavelmente não é uniforme — há concentração
  em valores específicos (ex: 44h exatas por convenção coletiva).
  Sem microdados intrasetoriais por hora exata, não é possível
  calcular a distribuição real.

Decisão: documentar o viés encontrado em DECISIONS.md → D16
  e adotar a formulação mais adequada como padrão.
""")